In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
device = "cuda"

In [2]:
from pathlib import Path
import urllib.request

def download_shakespeare_text():
    path = Path("datasets/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
    url = "https://homl.info/shakespeare"
    urllib.request.urlretrieve(url, path)
    return path.read_text()

shakespeare_text = download_shakespeare_text()

In [3]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [4]:
vocab = sorted(set(shakespeare_text.lower()))
"".join(vocab)

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

In [5]:
char_to_id = {char:index for index, char in enumerate(vocab)}
id_to_char = {index:char for index, char in enumerate(vocab)}
print(char_to_id["a"], id_to_char[13])

13 a


In [6]:
import torch

def encode_text(text):
    return torch.tensor([char_to_id[char] for char in text.lower()])

def decode_text(char_ids):
    return "".join([id_to_char[char_id.item()] for char_id in char_ids])

In [7]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text)
        self.window_length = window_length
    
    def __len__(self):
        return len(self.encoded_text) - self.window_length
    
    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset out of range")
        end = idx + self.window_length
        window = self.encoded_text[idx:end]
        target = self.encoded_text[idx+1:end+1]
        return window, target

In [8]:
window_length = 50
batch_size = 512

train_set = CharDataset(shakespeare_text[:1000000], window_length)
valid_set = CharDataset(shakespeare_text[1000000:1060000], window_length)
test_set = CharDataset(shakespeare_text[1060000:], window_length)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, pin_memory=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=batch_size, pin_memory=True)

# Embedding

In [9]:
import torch.nn as nn
torch.manual_seed(42)
embed = nn.Embedding(5,3)
embed(torch.tensor([[3,2], [0,2]]))

tensor([[[ 0.2674,  0.5349,  0.8094],
         [ 2.2082, -0.6380,  0.4617]],

        [[ 0.3367,  0.1288,  0.2345],
         [ 2.2082, -0.6380,  0.4617]]], grad_fn=<EmbeddingBackward0>)

In [10]:
class ShakespeareModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, X):
        embeddings = self.embed(X)
        outputs, _states = self.gru(embeddings)
        return self.output(outputs).permute(0,2,1)

torch.manual_seed(42)
model = ShakespeareModel(len(vocab)).to(device)

In [11]:
from torch.optim import Optimizer
from torchmetrics.classification import MulticlassAccuracy

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss = nn.CrossEntropyLoss()

def train(model:nn.Module, optimizer:Optimizer,train_loader:DataLoader, 
          valid_loader:DataLoader, loss_fn:nn.Module = nn.CrossEntropyLoss(), epoch:int = 1):
    for i in range(epoch):
        model.train()
        optimizer.zero_grad()
        model.train()
        train_loss, train_correct, train_total = 0,0,0
        for idx, (X,y) in enumerate(train_loader):
            X,y = X.to(device), y.to(device)
            y_pred = model(X)
            loss = loss_fn(y_pred, y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            train_loss += loss.item()
            train_correct += (y_pred.argmax(dim=1) == y).sum().item()
            train_total += y.numel()
        
        
        #eval part------------
        model.eval()
        val_loss, val_correct, val_total = 0,0,0
        
        with torch.no_grad():
            for X,y in valid_loader:
                X,y = X.to(device), y.to(device)
                y_pred = model(X)
                val_loss = loss_fn(y_pred, y)
                val_correct += (y_pred.argmax(dim=1) == y).sum().item()
                val_total += y.numel()
                
        print(f"Epoch {epoch+1}/{i} | "
              f"Train loss: {train_loss/len(train_loader):.4f}, acc: {train_correct/train_total:.4f} | "
              f"Val loss: {val_loss/len(valid_loader):.4f}, acc: {val_correct/val_total:.4f}")

def test(model:nn.Module, loss_fn:nn.Module, test_loader:DataLoader):
    model.eval()
    metric = MulticlassAccuracy(len(vocab), average="micro").to(device)
    total_metric:float = 0
    total_loss:float = 0
    
    with torch.no_grad():
        for X,y in test_loader:
            X,y = X.to(device), y.to(device)
            y_pred = model(X)
            loss = loss_fn(y_pred, y)
            total_loss += loss.item()
            total_metric += metric(y_pred, y)
        
        print(f"loss_fn:{total_loss/len(test_loader)}, acc:{total_metric/len(test_loader)}")


In [24]:
train(model, optimizer, train_loader, valid_loader, loss, 5)

Epoch 6/0 | Train loss: 1.4235, acc: 0.5579 | Val loss: 0.0102, acc: 0.5431
Epoch 6/1 | Train loss: 1.3835, acc: 0.5676 | Val loss: 0.0102, acc: 0.5487
Epoch 6/2 | Train loss: 1.3634, acc: 0.5725 | Val loss: 0.0102, acc: 0.5482
Epoch 6/3 | Train loss: 1.3508, acc: 0.5756 | Val loss: 0.0104, acc: 0.5500
Epoch 6/4 | Train loss: 1.3419, acc: 0.5779 | Val loss: 0.0103, acc: 0.5489


In [13]:
model.eval()
text = "To be or not to b"
encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
with torch.no_grad():
    Y_logits = model(encoded_text)
    predicted_char_id = Y_logits[0, :, -1].argmax().item()
    predicted_char = id_to_char[predicted_char_id]

In [25]:
predicted_char

'e'

In [26]:
torch.manual_seed(42)
probs = torch.tensor([[0.5, 0.4, 0.1]])
samples = torch.multinomial(probs, replacement=True, num_samples=8)
samples

tensor([[0, 0, 0, 0, 1, 0, 2, 2]])

In [27]:
import torch.nn.functional as F

def next_char(model:nn.Module, text, temperature=1):
    encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
    with torch.no_grad():
        Y_logits = model(encoded_text)
        Y_probas = F.softmax(Y_logits[0, :, -1]/temperature, dim=-1)
        predicted_char_id = torch.multinomial(Y_probas, num_samples=1).item()
    return id_to_char[predicted_char_id]

def extend_text(model:nn.Module, text, n_chars:int = 80, temperature = 1):
    for _ in range(n_chars):
        text += next_char(model, text, temperature)
    return text

In [29]:
print(extend_text(model, "To be or not to b", temperature=1))

To be or not to be, and repeal thy grace
have too respected by an expiris person; he fear, move
t
